# eval.ipynb
## Made in Rwanda Recommender — Evaluation
**AIMS KTT Hackathon · S2.T1.3**

Computes **NDCG@5** and **local-presence rate** on `data/queries.csv`.

In [ ]:
# Install dependencies (Colab)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                        'pandas', 'numpy', 'scikit-learn', 'scipy'])

In [ ]:
import os, sys
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import rankdata

# Add repo root to path so we can import recommender
repo_root = os.path.abspath('.')
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from recommender import MadeInRwandaRecommender
print('Imports OK')

In [ ]:
# Load data
catalog = pd.read_csv('data/catalog.csv')
queries = pd.read_csv('data/queries.csv')
click_log = pd.read_csv('data/click_log.csv')

print(f'Catalog: {len(catalog)} products')
print(f'Queries: {len(queries)} queries')
print(f'Click log: {len(click_log)} events')
print()
print('Catalog columns:', list(catalog.columns))
print('Queries columns:', list(queries.columns))
print('Click log columns:', list(click_log.columns))

In [ ]:
# Build relevance labels from click_log
# relevance[query_id][sku] = click count (higher = more relevant)
relevance = {}
for _, row in click_log.iterrows():
    qid = row['query_id']
    sku = row['sku']
    relevance.setdefault(qid, {})
    relevance[qid][sku] = relevance[qid].get(sku, 0) + 1

print(f'Queries with click relevance: {len(relevance)}')

In [ ]:
def ndcg_at_k(recommended_skus, relevant_dict, k=5):
    """Compute NDCG@k for a single query."""
    dcg = 0.0
    for i, sku in enumerate(recommended_skus[:k]):
        rel = relevant_dict.get(sku, 0)
        dcg += rel / np.log2(i + 2)
    # Ideal DCG
    ideal_rels = sorted(relevant_dict.values(), reverse=True)[:k]
    idcg = sum(r / np.log2(i + 2) for i, r in enumerate(ideal_rels))
    return dcg / idcg if idcg > 0 else 0.0

print('NDCG@k function defined')

In [ ]:
# Initialise recommender
rec = MadeInRwandaRecommender()

RWANDAN_DISTRICTS = {
    'kigali', 'gasabo', 'kicukiro', 'nyarugenge', 'nyamirambo',
    'huye', 'musanze', 'rubavu', 'rusizi', 'nyagatare',
    'rwamagana', 'kayonza', 'kirehe', 'ngoma', 'bugesera',
    'kamonyi', 'muhanga', 'ruhango', 'nyanza', 'gisagara',
    'nyaruguru', 'nyamagabe', 'karongi', 'rutsiro', 'ngororero',
    'nyabihu', 'gakenke', 'rulindo', 'gicumbi', 'burera', 'gatsibo'
}

results = []

for _, qrow in queries.iterrows():
    qid = qrow.get('query_id', qrow.name)
    q_text = str(qrow.get('query_text', qrow.get('query', '')))
    
    recs = rec.recommend(q_text, top_k=5)
    rec_skus = recs['sku'].tolist()
    
    # NDCG@5
    rel_dict = relevance.get(qid, {})
    ndcg5 = ndcg_at_k(rec_skus, rel_dict, k=5)
    
    # Local-presence: at least 1 local in top-3
    top3_local = recs.head(3)['is_local'].any() if 'is_local' in recs.columns else False
    
    results.append({
        'query_id': qid,
        'query_text': q_text,
        'ndcg_at_5': ndcg5,
        'local_in_top3': int(top3_local),
        'top5_skus': ','.join(map(str, rec_skus))
    })

results_df = pd.DataFrame(results)
print(results_df.head(10))

In [ ]:
# Summary metrics
mean_ndcg = results_df['ndcg_at_5'].mean()
local_presence_rate = results_df['local_in_top3'].mean()

summary = pd.DataFrame([{
    'metric': 'NDCG@5 (mean)',
    'value': round(mean_ndcg, 4)
}, {
    'metric': 'Local-presence rate (top-3)',
    'value': round(local_presence_rate, 4)
}])

print('=== EVALUATION SUMMARY ===')
print(summary.to_string(index=False))
print()
print(f'NDCG@5            : {mean_ndcg:.4f}')
print(f'Local-presence rate: {local_presence_rate:.4f} ({local_presence_rate*100:.1f}%)')

In [ ]:
# Save results
results_df.to_csv('evaluation_results.csv', index=False)
summary.to_csv('evaluation_summary.csv', index=False)
print('Saved evaluation_results.csv and evaluation_summary.csv')